In [ ]:
import phoebe
import json


fnames = ['true.bundle', 'true_mcmc_physical.bundle', 'true_mcmc_sum_ratios.bundle', 'true_mcmc_rvs.bundle']
i = 2
fname = fnames[i]

f = open(fname, 'r')
params = json.load(f)

if fname=='true.bundle':
    b = phoebe.default_binary()
    b.flip_constraint('mass@primary', solve_for='sma@binary')
    b.flip_constraint('mass@secondary', solve_for='q@binary')
if fname=='true_mcmc_physical.bundle':
    b = phoebe.load('true.bundle')

    b.add_dataset('lc', times=[0], fluxes=[1], sigmas=[1])
    b.run_compute()
    b.add_distribution({
        'requiv@primary': phoebe.gaussian_around(0.1),
        'requiv@secondary': phoebe.gaussian_around(0.1),
        'mass@primary': phoebe.gaussian_around(0.5),
        'mass@secondary': phoebe.gaussian_around(0.5),
        'teff@primary': phoebe.gaussian_around(250),
        'teff@secondary': phoebe.gaussian_around(250)
    }, distribution='physical', overwrite_all=True)
    b.add_solver('sampler.emcee', solver='mcmc_physical',
                 init_from='physical',
                 compute='phoebe01', nwalkers=16, niters=2, progress_every_niters=0)
    b.add_server('remoteslurm', crimpl_name='terra', nprocs=2, walltime=2,
                 use_conda=True, conda_env='phoebe-workshop',
                 server='terra')
    b.run_solver(solution='mcmc_physical_solution')
if fname=='true_mcmc_sum_ratios.bundle':
    b = phoebe.load('true.bundle')

    b.add_dataset('lc', times=[0], fluxes=[1], sigmas=[1])
    b.run_compute()
    
    b.flip_constraint('q', solve_for='mass@secondary')
    b.flip_constraint('sma@binary', solve_for='mass@primary') # optional since lc also doesn't constrain sma
    
    b.flip_constraint('requivsumfrac', solve_for='requiv@primary@component')
    b.flip_constraint('requivratio', solve_for='requiv@secondary@component')
    b.flip_constraint('teffratio', solve_for='teff@secondary@component')
    
    b.add_distribution({
            'requivsumfrac': phoebe.gaussian_around(0.01),
            'requivratio': phoebe.gaussian_around(0.1),
            'teffratio': phoebe.gaussian_around(0.1),
            'q': phoebe.gaussian_around(0.1)
        }, distribution='sum_ratios', overwrite_all=True)
    b.add_solver('sampler.emcee', solver='mcmc_sum_ratios',
                 init_from='sum_ratios',
                 compute='phoebe01', nwalkers=16, niters=2, progress_every_niters=0)
    b.add_server('remoteslurm', crimpl_name='terra', nprocs=2, walltime=2,
                 use_conda=True, conda_env='phoebe-workshop',
                 server='terra')
    b.run_solver(solution='mcmc_sum_ratios_solution')
if fname == 'true_mcmc_rvs.bundle':
    b = phoebe.load('true.bundle')
    b.add_dataset('rv', times=[0], rvs=[0], sigmas=[1])
    b.run_compute()

    b.flip_constraint('q', solve_for='mass@secondary')
    b.flip_constraint('sma@binary', solve_for='mass@primary')
    
    b.add_distribution(
        {'q': phoebe.gaussian_around(0.1),
            'vgamma': phoebe.gaussian_around(1),
            'sma@binary': phoebe.gaussian_around(0.5)
        },
        distribution='dist_rv1')

    b.add_solver('sampler.emcee', solver='mcmc_rv1',
             init_from='dist_rv1',
             compute='phoebe01', nwalkers=16, niters=2, progress_every_niters=0)
    b.add_solver('sampler.emcee', solver='mcmc_rvs',
                 init_from='dist_rv1',
                 compute='phoebe01', nwalkers=16, niters=2, progress_every_niters=0)
    b.add_server('remoteslurm', crimpl_name='terra', nprocs=2, walltime=2,
                 use_conda=True, conda_env='phoebe-workshop',
                 server='terra')
    b.run_solver('mcmc_rv1', solution='mcmc_rv1_solution')
    b.run_solver('mcmc_rvs', solution='mcmc_rvs_solution')


tags = list(b.tags.keys())

uniqueid_maps = {}
for param in params:
    filter_kwargs = {k:v if v!='clusty' else 'terra' for k,v in param.items() if k+"s" in tags}
    value = param.get('value')
    if filter_kwargs['qualifier'] in ['detached_job', 'phoebe_version']:
        continue
    if filter_kwargs.get('solution') == 'mcmc_rv1_solution' and filter_kwargs.get('solver') == 'mcmc_rvs':
        continue
    try:
        b_param = b.get_parameter(check_visible=False, check_default=False, **filter_kwargs)
    except ValueError:
        print("*** skipping: ", filter_kwargs, value)
        continue
    if not b_param.is_constraint:
        b_param.set_value(value=value, ignore_readonly=True)
        uniqueid_maps[b_param._uniqueid] = param.get('uniqueid')
b.save(fname)

with open(fname, 'r') as f:
    content = f.read()

for old_uid, new_uid in uniqueid_maps.items():
    content = content.replace(f'"{old_uid}"', f'"{new_uid}"')

with open(fname, 'w') as f:
    f.write(content)

b = phoebe.load(fname)

print(f"*** COMPLETED {fname} ***")